# **Cell 1 — Setup: Mount Drive, Seed, Define Paths**

In [1]:
!pip install -q transformers accelerate sentencepiece

from google.colab import drive
drive.mount('/content/drive')

from transformers import set_seed
set_seed(42)

import os
PROJECT_ROOT = '/content/drive/MyDrive/Bengali Paraphrasing'
PROCESSED_DIR = f'{PROJECT_ROOT}/processed_data'
OUTPUT_DIR = f'{PROJECT_ROOT}/outputs/generations'
os.makedirs(OUTPUT_DIR, exist_ok=True)

Mounted at /content/drive


# **Cell 2 — Load the Full, Untouched Test Set**

In [2]:
import pandas as pd

test_df = pd.read_parquet(os.path.join(PROCESSED_DIR, 'test_clean.parquet'))
print(f"Full test set size: {len(test_df)}")  # should be 23,332
print(test_df.head(3))

Full test set size: 23332
                                              source  \
0               কিছুদিন আগে প্যারিস থেকে ঘুরে এসেছি।   
1  ভাড়া করে ফেললেন কার্নেগি হলের মতো অত্যন্ত অভি...   
2  সম্পূর্ণ নিয়ন্ত্রণ হারিয়ে জাহাজ পড়লো ঘোর সম...   

                                              target  
0                   কয়েকদিন আগে আমি প্যারিসে ছিলাম।  
1  তিনি কার্নেগী হলের মতো একটি অভিজাত মিলনায়তন ভ...  
2  জাহাজটি তার সম্পূর্ণ নিয়ন্ত্রণ হারিয়ে গভীর স...  


# **Cell 3 — Load IndicBARTSS (best_model)**

In [3]:
import torch
from transformers import AlbertTokenizer, AutoModelForSeq2SeqLM
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IB_MODEL_DIR = f'{PROJECT_ROOT}/checkpoints/indicbartss/best_model'
IB_SPIECE_PATH = f'{PROJECT_ROOT}/base_model_cache/indicbartss/spiece.model'

ib_tokenizer = AlbertTokenizer.from_pretrained(IB_MODEL_DIR, do_lower_case=False, use_fast=False, keep_accents=True)
ib_model = AutoModelForSeq2SeqLM.from_pretrained(IB_MODEL_DIR, attn_implementation="eager").to(device)
ib_model.eval()

sp = spm.SentencePieceProcessor()
sp.load(IB_SPIECE_PATH)

BN_TAG = "<2bn>"
bn_tag_id = ib_tokenizer._convert_token_to_id_with_added_voc(BN_TAG)
eos_id = ib_tokenizer.convert_tokens_to_ids("</s>")
IB_MAX_LENGTH = 96

def encode_sp(text):
    pieces = sp.encode(text, out_type=str)
    return ib_tokenizer.convert_tokens_to_ids(pieces)

# Documentation: params + version
total_params = sum(p.numel() for p in ib_model.parameters())
print(f"IndicBARTSS total parameters: {total_params:,}")
import json
with open(f'{IB_MODEL_DIR}/config.json') as f:
    print("transformers_version:", json.load(f).get('transformers_version', 'not recorded'))

Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]

IndicBARTSS total parameters: 244,018,176
transformers_version: 5.13.1


# **Cell 4 — Generate with IndicBARTSS (full test set, batch 16, greedy, resume-safe)**

In [4]:
IB_OUTPUT_PATH = f'{OUTPUT_DIR}/indicbartss_generations.parquet'
batch_size = 16

if os.path.exists(IB_OUTPUT_PATH):
    ib_results_df = pd.read_parquet(IB_OUTPUT_PATH)
    done_count = len(ib_results_df)
    print(f"Resuming: {done_count} examples already generated.")
else:
    ib_results_df = pd.DataFrame(columns=['source', 'target', 'generated'])
    done_count = 0

remaining_df = test_df.iloc[done_count:].reset_index(drop=True)

import time
new_rows = []
save_every = 50  # batches, i.e. periodic safety checkpointing to Drive

for batch_num, i in enumerate(range(0, len(remaining_df), batch_size)):
    batch = remaining_df.iloc[i:i+batch_size]
    input_ids_list = [(encode_sp(s) + [eos_id, bn_tag_id])[:IB_MAX_LENGTH] for s in batch['source']]
    max_len = max(len(x) for x in input_ids_list)
    padded = [x + [ib_tokenizer.pad_token_id]*(max_len-len(x)) for x in input_ids_list]
    attn = [[1]*len(x) + [0]*(max_len-len(x)) for x in input_ids_list]
    input_tensor = torch.tensor(padded).to(device)
    attn_tensor = torch.tensor(attn).to(device)

    with torch.no_grad():
        output_ids = ib_model.generate(input_ids=input_tensor, attention_mask=attn_tensor,
                                        decoder_start_token_id=bn_tag_id, max_length=IB_MAX_LENGTH, do_sample=False)

    for j, out_ids in enumerate(output_ids):
        generated = ib_tokenizer.decode(out_ids, skip_special_tokens=True)
        new_rows.append({
            'source': batch['source'].iloc[j],
            'target': batch['target'].iloc[j],
            'generated': generated
        })

    if (batch_num + 1) % save_every == 0:
        ib_results_df = pd.concat([ib_results_df, pd.DataFrame(new_rows)], ignore_index=True)
        ib_results_df.to_parquet(IB_OUTPUT_PATH, index=False)
        new_rows = []
        print(f"Checkpoint saved: {len(ib_results_df)} / {len(test_df)} generated.")

# Final save of any remaining rows
if new_rows:
    ib_results_df = pd.concat([ib_results_df, pd.DataFrame(new_rows)], ignore_index=True)
    ib_results_df.to_parquet(IB_OUTPUT_PATH, index=False)

print(f"IndicBARTSS generation complete: {len(ib_results_df)} / {len(test_df)}")

Checkpoint saved: 800 / 23332 generated.
Checkpoint saved: 1600 / 23332 generated.
Checkpoint saved: 2400 / 23332 generated.
Checkpoint saved: 3200 / 23332 generated.
Checkpoint saved: 4000 / 23332 generated.
Checkpoint saved: 4800 / 23332 generated.
Checkpoint saved: 5600 / 23332 generated.
Checkpoint saved: 6400 / 23332 generated.
Checkpoint saved: 7200 / 23332 generated.
Checkpoint saved: 8000 / 23332 generated.
Checkpoint saved: 8800 / 23332 generated.
Checkpoint saved: 9600 / 23332 generated.
Checkpoint saved: 10400 / 23332 generated.
Checkpoint saved: 11200 / 23332 generated.
Checkpoint saved: 12000 / 23332 generated.
Checkpoint saved: 12800 / 23332 generated.
Checkpoint saved: 13600 / 23332 generated.
Checkpoint saved: 14400 / 23332 generated.
Checkpoint saved: 15200 / 23332 generated.
Checkpoint saved: 16000 / 23332 generated.
Checkpoint saved: 16800 / 23332 generated.
Checkpoint saved: 17600 / 23332 generated.
Checkpoint saved: 18400 / 23332 generated.
Checkpoint saved: 19200 

In [5]:
ib_results_df.head(10)          # first 10 rows
ib_results_df.sample(5, random_state=42)   # 5 random rows instead

,source,target,generated
2864,"দীর্ঘ ভ্রমণে ক্লান্ত সবাই, সকালেই রওনা দিতে হব...","দীর্ঘ যাত্রায় ক্লান্ত, সকালে আমাদের চলে যেতে ...","<2bn> দীর্ঘ ভ্রমণ শেষে, সবাই সকালে আবার ফিরে আ..."
7638,এ বাড়িতে সাধারণত কোনো ঝগড়াঝাঁটি হয় না।,এ বাড়িতে কোনো ঝগড়া নেই।,<2bn> এ বাড়িতে সাধারণত কোনো ঝগড়া হয় না।</s>
12343,কিছু ভিডিওতে কল্পিত শত্রুর বিরুদ্ধে বদলা নেওয়...,কিছু ভিডিও কাল্পনিক শত্রুদের বিরুদ্ধে প্রতিশোধ...,<2bn> কিছু ভিডিওতে শত্রুর বিরুদ্ধে শত্রুর বিরু...
21159,প্রথম দিনে মোট ২৬ জনকে টিকা দেয়া হয়।,প্রথম দিন সর্বমোট ২৬ জনকে অনাক্রম্য করা হয়।,<2bn> প্রথম দিনে ২৬ জন টিকা প্রদান করা হয়।</s>
22798,দেসিয়াস তখন শত্রুদলের মধ্যে ঝাঁপ দেন।,এরপর দেসিয়াস শত্রুদের মধ্যে ঝাঁপিয়ে পড়েছিলেন।,<2bn> দেসিয়াস শত্রুদের বিরুদ্ধে যুদ্ধে ঝাঁপিয...


In [6]:
IB_OUTPUT_PATH = f'{OUTPUT_DIR}/indicbartss_generations.parquet'
ib_results_df = pd.read_parquet(IB_OUTPUT_PATH)

def clean_generated_text(text):
    text = text.replace(BN_TAG, "").replace("</s>", "").strip()
    return text

ib_results_df['generated'] = ib_results_df['generated'].apply(clean_generated_text)
ib_results_df.to_parquet(IB_OUTPUT_PATH, index=False)

print("Cleaned and re-saved.")
ib_results_df.sample(5, random_state=42)

Cleaned and re-saved.


,source,target,generated
2864,"দীর্ঘ ভ্রমণে ক্লান্ত সবাই, সকালেই রওনা দিতে হব...","দীর্ঘ যাত্রায় ক্লান্ত, সকালে আমাদের চলে যেতে ...","দীর্ঘ ভ্রমণ শেষে, সবাই সকালে আবার ফিরে আসবে।"
7638,এ বাড়িতে সাধারণত কোনো ঝগড়াঝাঁটি হয় না।,এ বাড়িতে কোনো ঝগড়া নেই।,এ বাড়িতে সাধারণত কোনো ঝগড়া হয় না।
12343,কিছু ভিডিওতে কল্পিত শত্রুর বিরুদ্ধে বদলা নেওয়...,কিছু ভিডিও কাল্পনিক শত্রুদের বিরুদ্ধে প্রতিশোধ...,কিছু ভিডিওতে শত্রুর বিরুদ্ধে শত্রুর বিরুদ্ধে য...
21159,প্রথম দিনে মোট ২৬ জনকে টিকা দেয়া হয়।,প্রথম দিন সর্বমোট ২৬ জনকে অনাক্রম্য করা হয়।,প্রথম দিনে ২৬ জন টিকা প্রদান করা হয়।
22798,দেসিয়াস তখন শত্রুদলের মধ্যে ঝাঁপ দেন।,এরপর দেসিয়াস শত্রুদের মধ্যে ঝাঁপিয়ে পড়েছিলেন।,দেসিয়াস শত্রুদের বিরুদ্ধে যুদ্ধে ঝাঁপিয়ে পড়ে।


# **Cell 5 — Load BanglaT5 (best_model)**

In [8]:
!pip install git+https://github.com/csebuetnlp/normalizer

from transformers import AutoTokenizer
from normalizer import normalize

BT5_MODEL_DIR = f'{PROJECT_ROOT}/checkpoints/banglat5/best_model'

bt5_tokenizer = AutoTokenizer.from_pretrained(BT5_MODEL_DIR, use_fast=False)
bt5_model = AutoModelForSeq2SeqLM.from_pretrained(BT5_MODEL_DIR).to(device)
bt5_model.eval()

BT5_MAX_LENGTH = 96

# Documentation: params + version
total_params = sum(p.numel() for p in bt5_model.parameters())
print(f"BanglaT5 total parameters: {total_params:,}")
with open(f'{BT5_MODEL_DIR}/config.json') as f:
    print("transformers_version:", json.load(f).get('transformers_version', 'not recorded'))

  Cloning https://github.com/csebuetnlp/normalizer to /tmp/pip-req-build-7mp7_2y7
  Running command git clone --filter=blob:none --quiet https://github.com/csebuetnlp/normalizer /tmp/pip-req-build-7mp7_2y7
  Resolved https://github.com/csebuetnlp/normalizer to commit d405944dde5ceeacb7c2fd3245ae2a9dea5f35c9
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 17.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for normalizer: filename=normalizer-0.0.1-py3-none-any.whl size=6860 sha256=bd1788e19c4bb6e2021eed814f4de4d68ef5cb769d546a61f10dac3445b7786e
  Stored in directory: /tmp/pip-ephem-wheel-cache-t95hfa78/wheels/f9/d8/55/a13fa77440d3e80bf10ff80176ba67c7a0543a67827ef0b8eb
  Created wheel for emoji: filename=emoji-1.4.2-py3-none-any.whl size=186456 sha256=e06c0ce68922205ede93ed28153b50798cf4da05

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


BanglaT5 total parameters: 247,577,856
transformers_version: 5.13.1


# **Cell 6 — Generate with BanglaT5 (full test set, batch 16, greedy, resume-safe)**

In [9]:
BT5_OUTPUT_PATH = f'{OUTPUT_DIR}/banglat5_generations.parquet'
batch_size = 16

if os.path.exists(BT5_OUTPUT_PATH):
    bt5_results_df = pd.read_parquet(BT5_OUTPUT_PATH)
    done_count = len(bt5_results_df)
    print(f"Resuming: {done_count} examples already generated.")
else:
    bt5_results_df = pd.DataFrame(columns=['source', 'target', 'generated'])
    done_count = 0

remaining_df = test_df.iloc[done_count:].reset_index(drop=True)

new_rows = []
save_every = 50

for batch_num, i in enumerate(range(0, len(remaining_df), batch_size)):
    batch = remaining_df.iloc[i:i+batch_size]
    batch_sources = [normalize(s) for s in batch['source'].tolist()]
    inputs = bt5_tokenizer(batch_sources, return_tensors="pt", padding=True, truncation=True, max_length=BT5_MAX_LENGTH).to(device)

    with torch.no_grad():
        output_ids = bt5_model.generate(**inputs, max_length=BT5_MAX_LENGTH, do_sample=False)

    for j, out_ids in enumerate(output_ids):
        generated = bt5_tokenizer.decode(out_ids, skip_special_tokens=True)
        new_rows.append({
            'source': batch['source'].iloc[j],
            'target': batch['target'].iloc[j],
            'generated': generated
        })

    if (batch_num + 1) % save_every == 0:
        bt5_results_df = pd.concat([bt5_results_df, pd.DataFrame(new_rows)], ignore_index=True)
        bt5_results_df.to_parquet(BT5_OUTPUT_PATH, index=False)
        new_rows = []
        print(f"Checkpoint saved: {len(bt5_results_df)} / {len(test_df)} generated.")

if new_rows:
    bt5_results_df = pd.concat([bt5_results_df, pd.DataFrame(new_rows)], ignore_index=True)
    bt5_results_df.to_parquet(BT5_OUTPUT_PATH, index=False)

print(f"BanglaT5 generation complete: {len(bt5_results_df)} / {len(test_df)}")

Checkpoint saved: 800 / 23332 generated.
Checkpoint saved: 1600 / 23332 generated.
Checkpoint saved: 2400 / 23332 generated.
Checkpoint saved: 3200 / 23332 generated.
Checkpoint saved: 4000 / 23332 generated.
Checkpoint saved: 4800 / 23332 generated.
Checkpoint saved: 5600 / 23332 generated.
Checkpoint saved: 6400 / 23332 generated.
Checkpoint saved: 7200 / 23332 generated.
Checkpoint saved: 8000 / 23332 generated.
Checkpoint saved: 8800 / 23332 generated.
Checkpoint saved: 9600 / 23332 generated.
Checkpoint saved: 10400 / 23332 generated.
Checkpoint saved: 11200 / 23332 generated.
Checkpoint saved: 12000 / 23332 generated.
Checkpoint saved: 12800 / 23332 generated.
Checkpoint saved: 13600 / 23332 generated.
Checkpoint saved: 14400 / 23332 generated.
Checkpoint saved: 15200 / 23332 generated.
Checkpoint saved: 16000 / 23332 generated.
Checkpoint saved: 16800 / 23332 generated.
Checkpoint saved: 17600 / 23332 generated.
Checkpoint saved: 18400 / 23332 generated.
Checkpoint saved: 19200 

In [10]:
bt5_results_df.head(10)          # first 10 rows
bt5_results_df.sample(5, random_state=42)   # 5 random rows instead

,source,target,generated
2864,"দীর্ঘ ভ্রমণে ক্লান্ত সবাই, সকালেই রওনা দিতে হব...","দীর্ঘ যাত্রায় ক্লান্ত, সকালে আমাদের চলে যেতে ...","দীর্ঘ ভ্রমণের কারণে ক্লান্ত সবাই, সকালে রওনা দ..."
7638,এ বাড়িতে সাধারণত কোনো ঝগড়াঝাঁটি হয় না।,এ বাড়িতে কোনো ঝগড়া নেই।,এ বাড়িতে সাধারণত কোনো ঝগড়া হয় না।
12343,কিছু ভিডিওতে কল্পিত শত্রুর বিরুদ্ধে বদলা নেওয়...,কিছু ভিডিও কাল্পনিক শত্রুদের বিরুদ্ধে প্রতিশোধ...,কিছু ভিডিওতে কাল্পনিক শত্রুর বিরুদ্ধে প্রতিশোধ...
21159,প্রথম দিনে মোট ২৬ জনকে টিকা দেয়া হয়।,প্রথম দিন সর্বমোট ২৬ জনকে অনাক্রম্য করা হয়।,প্রথম দিনে ২৬ জনকে টিকা দেওয়া হয়।
22798,দেসিয়াস তখন শত্রুদলের মধ্যে ঝাঁপ দেন।,এরপর দেসিয়াস শত্রুদের মধ্যে ঝাঁপিয়ে পড়েছিলেন।,দেসিয়াস তখন শত্রু দলের মধ্যে ঝাঁপ দেন।


# ***DOCUMENTATION***

In [11]:
seq2seq_documentation = {
    "IndicBARTSS": {
        "total_params": sum(p.numel() for p in ib_model.parameters()),
        "transformers_version": json.load(open(f'{IB_MODEL_DIR}/config.json')).get('transformers_version'),
    },
    "BanglaT5": {
        "total_params": sum(p.numel() for p in bt5_model.parameters()),
        "transformers_version": json.load(open(f'{BT5_MODEL_DIR}/config.json')).get('transformers_version'),
    },
}

doc_path = f'{OUTPUT_DIR}/seq2seq_model_documentation.json'
with open(doc_path, 'w', encoding='utf-8') as f:
    json.dump(seq2seq_documentation, f, ensure_ascii=False, indent=2)
print(json.dumps(seq2seq_documentation, ensure_ascii=False, indent=2))

{
  "IndicBARTSS": {
    "total_params": 244018176,
    "transformers_version": "5.13.1"
  },
  "BanglaT5": {
    "total_params": 247577856,
    "transformers_version": "5.13.1"
  }
}
